===============================================================================================

$$\text{This script Reproducible synthetic identifiability profiles on a 5-unit gamma grid}$$
$$\text{Avulundiah Edwin Phiri}$$
$$\text{Department of Mathematics, Copperbelt University, Kitwe, Zambia}$$
$$\text{31/07/2026}$$

==============================================================================================


In [2]:
"""Reproducible synthetic identifiability profiles on a 5-unit gamma grid.

This is an experimental-design calculation, not empirical calibration. The
remaining parameters and the noise standard deviations are treated as known.
Four independent Gaussian-noise realizations (fixed seeds) probe the stability
of the profile interval; the reported grid intervals are not exact CIs.
"""
from pathlib import Path
import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
from scipy.stats import chi2

ROOT=Path(__file__).resolve().parent
KAPPA=.3; LAM=.35; N=5.; X0=.2; S0=.9
TRUE_R=.16; TRUE_GAMMA=540.
T_K=np.linspace(0,15,31)
T_S=np.r_[0.0,.0005,.001,.002,.003,.005,.01,.02,.05,.25,np.linspace(.5,15,15)]
SIG_K=.008; SIG_S=.012
SEEDS=[20261122,20261123,20261124,20261125]
G_FINE=np.arange(400.,701.,5.)
G_K=np.unique(np.r_[np.arange(40.,301.,10.),np.arange(325.,1501.,75.),1500.])
CRIT=chi2.ppf(.95,1)

def solve_model(r,gamma):
    t=np.unique(np.r_[T_K,T_S]); c=1+X0
    def rhs(_,y):
        s,x=y; f=s/(KAPPA+s)
        return [gamma*(r*max(c-x,0)**N*(1-s)-f*x),(f-LAM)*x]
    sol=solve_ivp(rhs,(0.,float(t[-1])),(S0,X0),method='LSODA',t_eval=t,rtol=2e-8,atol=2e-10)
    if not sol.success: return None
    return (np.maximum(c-np.interp(T_K,t,sol.y[1]),0)**N,
            np.interp(T_S,t,sol.y[0]))

TRUE=solve_model(TRUE_R,TRUE_GAMMA)
records=[]; summary=[]
for i,seed in enumerate(SEEDS):
    rng=np.random.default_rng(seed)
    kobs=TRUE[0]+rng.normal(0,SIG_K,len(T_K))
    sobs=TRUE[1]+rng.normal(0,SIG_S,len(T_S))
    for design in ('k_only','k_and_early_effluent'):
        grid=G_K if design=='k_only' else G_FINE
        current=[]
        for gamma in grid:
            def objective(r):
                sol=solve_model(r,gamma)
                if sol is None:return 1e12
                k,s=sol
                obj=np.sum(((k-kobs)/SIG_K)**2)
                if design!='k_only': obj+=np.sum(((s-sobs)/SIG_S)**2)
                return obj
            opt=minimize_scalar(objective,bounds=(.06,.5),method='bounded',options={'xatol':2e-6,'maxiter':45})
            current.append((gamma,opt.x,opt.fun))
        curr=np.array(current); best=np.min(curr[:,2]); mask=curr[:,2]-best<=CRIT
        accepted=curr[mask,0]
        summary.append(dict(seed=seed,design=design,step_gamma=5 if design!='k_only' else 'coarse: 10 below 300, 75 above 325',
                            gamma_best=float(curr[np.argmin(curr[:,2]),0]),
                            gamma_low=float(accepted.min()) if len(accepted) else np.nan,
                            gamma_high=float(accepted.max()) if len(accepted) else np.nan,
                            upper_search_limit=float(curr[-1,0]),lower_search_limit=float(curr[0,0]),
                            truncated_upper=bool(len(accepted) and accepted.max()==curr[-1,0])))
        for gamma,r,loss in current:
            records.append(dict(seed=seed,design=design,gamma=gamma,best_r=r,chi_square=loss,
                                delta_chi_square=loss-best))
    print('seed',seed,[(d['design'],d['gamma_low'],d['gamma_high']) for d in summary[-2:]],flush=True)

with (ROOT/'identifiability_synthetic_profile.csv').open('w',newline='') as fh:
    w=csv.DictWriter(fh,fieldnames=records[0].keys());w.writeheader();w.writerows(records)
with (ROOT/'identifiability_profile_summary.csv').open('w',newline='') as fh:
    w=csv.DictWriter(fh,fieldnames=summary[0].keys());w.writeheader();w.writerows(summary)

fig,axs=plt.subplots(1,2,figsize=(10.4,4),sharey=True)
for ax,design,title in zip(axs,['k_only','k_and_early_effluent'],['Permeability only','Permeability + early effluent']):
    for z,seed in enumerate(SEEDS):
        xx=np.array([row['gamma'] for row in records if row['design']==design and row['seed']==seed])
        yy=np.array([row['delta_chi_square'] for row in records if row['design']==design and row['seed']==seed])
        ax.plot(xx,yy,lw=1.5,alpha=.85,label='Seed '+str(seed))
    ax.axhline(CRIT,c='0.35',ls='--',lw=1.2)
    ax.axvline(TRUE_GAMMA,c='0.45',ls=':',lw=1.1)
    ax.set(xlabel=r'Profiled $\gamma$',title=title,ylim=(0,15),xlim=(40,1500) if design=='k_only' else (400,700))
    ax.grid(alpha=.18)
axs[0].set_ylabel(r'$\Delta\chi^2$ (profile relative to minimum)')
axs[1].legend(loc='upper right',frameon=False,fontsize=8)
fig.tight_layout()
fig.savefig(ROOT/'identifiability_synthetic_profile.pdf')
plt.close(fig)


NameError: name '__file__' is not defined